# Label-efficient lung nodule segmentation: 3D Mamba-UNet + diffusion pretraining (LIDC-IDRI)

**Kaggle setup:** *Accelerator: GPU (T4 x1 or P100)*, *Internet: on*, and *Add Input →* **`washingtongold/lidcidri30`**,
the original TCIA LIDC-IDRI DICOM folders for about a third of the patients (41 GB).
(`justinkirby/the-cancer-imaging-archive-lidcidri` holds only the XML annotations, no CT images.)

This notebook runs the whole study:
1. It preprocesses every nodule annotated by ≥2 radiologists into 96³ cubes at 0.8 mm, keeping the 50% consensus mask and each reader's own mask.
2. **QA:** it re-creates the pylidc consensus tutorial figure for LIDC-IDRI-0078 and places our cube next to it, plus a numeric check against pylidc's native volumes.
3. It pretrains the network as a diffusion denoiser on unlabelled CT cubes.
4. It fine-tunes {UNet, Mamba-UNet, Mamba-UNet+pretraining} on 10/25/50/100% of the labelled training patients and tests them all on the same held-out patients.

Finished runs are skipped when the notebook is re-run.

In [ ]:
!git clone -q -b rebuild https://github.com/chrisjason0505/lung-nodule-segmentation-mamba-diffusion.git repo || (cd repo && git pull -q)
%cd repo
!pip install -q "setuptools<81" pylidc==0.2.3 mambapy pydicom scikit-image
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# optional fused CUDA kernels (3-5x faster Mamba). The pure-PyTorch scan is used automatically if this fails.
INSTALL_MAMBA_SSM = True
if INSTALL_MAMBA_SSM:
    !pip install -q causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -2
!python -c "from lungseg.models.ssm import backend_name; print('Mamba backend:', backend_name())"

In [ ]:
!python -m pytest -q tests

In [ ]:
DICOM_ROOT = '/kaggle/input'          # the indexer finds the series anywhere below this
DATA = '/kaggle/working/lidc_crops'
RUNS = '/kaggle/working/runs'
QA = '/kaggle/working/qa'
!python -m lungseg.data.prepare_lidc --dicom_root "$DICOM_ROOT" --out "$DATA" --workers 4 --min_readers 2

In [ ]:
!python -m lungseg.qa --dicom_root "$DICOM_ROOT" --data "$DATA" --out "$QA" --n 60
from IPython.display import Image, Markdown, display
for f in ['pylidc_tutorial_0078.png', 'crop_grid.png']:
    display(Image(f'{QA}/{f}'))

In [ ]:
# speed / memory probe (20 steps each) so the step budget below can be sized to the 12 h limit
!python -m lungseg.train --data "$DATA" --out /kaggle/working/probe_unet  --arch unet  --steps 20 --val_every 20 --no_test
!python -m lungseg.train --data "$DATA" --out /kaggle/working/probe_mamba --arch mamba --steps 20 --val_every 20 --no_test

In [ ]:
STEPS, PRE_STEPS = 4000, 4000
!python -m lungseg.experiments --data "$DATA" --runs "$RUNS" --preset core --steps $STEPS --pretrain_steps $PRE_STEPS

In [ ]:
display(Markdown(open(f'{RUNS}/summary.md').read()))
display(Image(f'{RUNS}/data_efficiency.png'))
best = f'{RUNS}/mamba_ddep_f1_s0/best.pt'
!python -m lungseg.visualize --data "$DATA" --ckpt "$best" --out /kaggle/working/predictions.png
display(Image('/kaggle/working/predictions.png'))